In [2]:
import pickle
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp
import h5py
import os
from tqdm import tqdm

root_path = "./replogle_concat.h5ad"
replogle_full = ad.read_h5ad(root_path)

In [3]:
replogle_full.obs

,gem_group,gene,gene_id,transcript,gene_transcript,sgID_AB,mitopercent,UMI_count,z_gemgroup_UMI,cell_line
cell_barcode,,,,,,,,,,
AAACCCAAGAATAGTC-3-hepg2,3,KIAA1143,ENSG00000163807,P1P2,4360_KIAA1143_P1P2_ENSG00000163807,KIAA1143_+_44803075.23-P1P2|KIAA1143_+_4480308...,0.114029,11234.0,-0.611091,hepg2
AAACCCAAGAGGTATT-55-hepg2,55,SEPHS2,ENSG00000179918,P1P2,7779_SEPHS2_P1P2_ENSG00000179918,SEPHS2_-_30457178.23-P1P2|SEPHS2_+_30457164.23...,0.165242,45146.0,0.208833,hepg2
AAACCCAAGAGTGACC-39-hepg2,39,POLR2H,ENSG00000163882,P1P2,6549_POLR2H_P1P2_ENSG00000163882,POLR2H_+_184081227.23-P1P2|POLR2H_-_184081237....,0.162209,20190.0,-0.025114,hepg2
AAACCCAAGATGGCAC-43-hepg2,43,TFAM,ENSG00000108064,P1P2,8832_TFAM_P1P2_ENSG00000108064,TFAM_+_60145205.23-P1P2|TFAM_-_60145223.23-P1P2,0.071596,23912.0,2.079665,hepg2
AAACCCAAGCAACAAT-16-hepg2,16,TMEM214,ENSG00000119777,P1P2,9042_TMEM214_P1P2_ENSG00000119777,TMEM214_-_27255873.23-P1P2|TMEM214_-_27255853....,0.109515,8282.0,-0.214576,hepg2
...,...,...,...,...,...,...,...,...,...,...
TTTGTTGTCTGATGGT-14-rpe1,14,SLC1A5,ENSG00000105281,P1,7976_SLC1A5_P1_ENSG00000105281,SLC1A5_+_47291839.23-P1|SLC1A5_+_47291829.23-P1,0.062634,15423.0,0.347911,rpe1
TTTGTTGTCTGCACCT-44-rpe1,44,MAX,ENSG00000125952,P1P2,4871_MAX_P1P2_ENSG00000125952,MAX_+_65569008.23-P1P2|MAX_-_65568906.23-P1P2,0.079629,22228.0,0.767826,rpe1
TTTGTTGTCTGGGCAC-32-rpe1,32,ATP6V0C,ENSG00000185883,P1P2,675_ATP6V0C_P1P2_ENSG00000185883,ATP6V0C_+_2564168.23-P1P2|ATP6V0C_-_2563995.23...,0.049527,12377.0,-0.004215,rpe1


In [16]:
replogle_full.var_names

Index(['NOC2L', 'HES4', 'ISG15', 'SDF4', 'B3GALT6', 'UBE2J2', 'ACAP3', 'PUSL1',
       'INTS11', 'DVL1',
       ...
       'MT-CO2', 'MT-ATP8', 'MT-ATP6', 'MT-CO3', 'MT-ND3', 'MT-ND4L', 'MT-ND4',
       'MT-ND5', 'MT-ND6', 'MT-CYB'],
      dtype='object', name='gene_name_index', length=6546)

In [18]:
hv = replogle_full.var["highly_variable"].to_numpy()
hvg_names = replogle_full.var_names[hv]

# 关键：用 obsm['X_hvg'] 作为新 X
X_hvg = replogle_full.obsm["X_hvg"]

# 构建新 AnnData
adata = ad.AnnData(
    X=X_hvg,
    obs=replogle_full.obs.copy(),
    var=replogle_full.var.loc[hvg_names].copy(),
)

# 设置正确的 var_names（保险起见）
adata.var_names = hvg_names

# 可选：把原来的 uns / obsm / layers 也按需带过去
adata.uns = adata.uns.copy()
adata.write_h5ad("./only_hvg/Replogle_only_hvg.h5ad")

In [22]:
adata.var_names

Index(['HES4', 'ISG15', 'MIB2', 'CEP104', 'ACOT7', 'VAMP3', 'RERE', 'ENO1',
       'SLC25A33', 'CLSTN1',
       ...
       'VAMP7', 'MT-ND2', 'MT-CO1', 'MT-ATP8', 'MT-ATP6', 'MT-CO3', 'MT-ND3',
       'MT-ND4L', 'MT-ND5', 'MT-ND6'],
      dtype='object', name='gene_name_index', length=2000)